In [40]:
import os
import torch
import json
from dotenv import load_dotenv
from collections import defaultdict
from typing import List, Dict
import time

In [41]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_groq import ChatGroq
from langchain_classic.chains import LLMChain
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, ValidationError

In [42]:
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
os.environ['HF_TOKEN']     = os.getenv("HF_TOKEN")

# Model rotation — when one hits daily token limit, falls back to next
MODEL_POOL = [
    "llama-3.1-8b-instant",       # fastest, lowest token cost
    "gemma2-9b-it",               # fallback 1
    "llama3-8b-8192",             # fallback 2
]
current_model_idx = 0

def get_llm():
    return ChatGroq(model=MODEL_POOL[current_model_idx], temperature=0.2)

llm = get_llm()
print(f"✅ LLM: {MODEL_POOL[current_model_idx]}")

session_dbs       = {}
session_histories = {}
session_flags     = {}
flag_counts       = defaultdict(lambda: {"Red": 0, "Yellow": 0, "Green": 0})

✅ LLM: llama-3.1-8b-instant


In [43]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# all-MiniLM-L6-v2: fast inference, 384-dim, great for semantic similarity
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)
print("✅ Embeddings loaded: all-MiniLM-L6-v2")

Using device: cuda
✅ Embeddings loaded: all-MiniLM-L6-v2


In [44]:
# Same schema and prompt template as Agent Lexa
class ClassificationResult(BaseModel):
    flag: str = Field(description="Must be Red, Yellow, or Green")
    text: str = Field(description="Actual text from the PDF chunk")
    summary: str = Field(description="One to 2 line summary of the chunk")
    reason: str = Field(description="Why this classification was chosen")
    recommendation: str = Field(description="Recommended action or next steps", default="")

parser = PydanticOutputParser(pydantic_object=ClassificationResult)

# Same prompt template as Agent Lexa
classification_prompt = PromptTemplate(
    template="""
You are an Tax audit compliance checker.
Classify the following audit report text into exactly one category: Red, Yellow, or Green.

TEXT FROM AUDIT REPORT (use this as primary reference):
{chunk}

Relevant Tax Law Context (optional, for guidance only):
{law_context}

Rules:
- Red = Clearly violates law (late filing, under-reported income, missing withholding)
- Yellow = Ambiguous, needs human review (unclear scanned text, missing data, incomplete info)
- Green = Fully compliant (filed on time, correct tax deducted)

Return JSON ONLY with keys: flag, text, summary, reason, recommendation
{text_instructions}
""",
    input_variables=["chunk", "law_context"],
    partial_variables={"text_instructions": parser.get_format_instructions()},
)

classifier_chain = LLMChain(llm=llm, prompt=classification_prompt)
print("✅ Classification chain ready")

✅ Classification chain ready


In [45]:
def _rotate_model():
    """Switch to next model in pool when rate-limited"""
    global llm, current_model_idx
    current_model_idx = (current_model_idx + 1) % len(MODEL_POOL)
    llm = get_llm()
    print(f"🔄 Switched to model: {MODEL_POOL[current_model_idx]}")


def classify_batch(chunks: List[str]) -> List[Dict]:
    """Classify multiple chunks in a single API call with model rotation on 429"""

    chunk_section = ""
    for i, chunk in enumerate(chunks, 1):
        chunk_section += f"[{i}] {chunk[:300].strip()}\n\n"

    batch_prompt = f"""You are a tax audit compliance classifier.
Classify each numbered chunk below as Red, Yellow, or Green.

Red    = clear violation (irregular payment, non-compliance, missing withholding, late filing)
Yellow = ambiguous / needs review (incomplete info, unclear text, missing data)
Green  = fully compliant (proper documentation, timely filing, correct deductions)

CHUNKS:
{chunk_section}
Return ONLY a JSON array — one object per chunk, in order:
[{{"id":1,"flag":"Red","summary":"...","reason":"...","recommendation":"..."}},...]
NO extra text."""

    # Try each model in pool before giving up
    for attempt in range(len(MODEL_POOL)):
        try:
            response = llm.invoke(batch_prompt)
            content  = response.content.strip()
            start = content.find('[')
            end   = content.rfind(']') + 1
            if start != -1 and end > start:
                parsed  = json.loads(content[start:end])
                results = []
                for i, item in enumerate(parsed):
                    orig = chunks[i] if i < len(chunks) else ""
                    results.append({
                        "flag":           item.get("flag", "Yellow"),
                        "text":           orig.strip(),
                        "summary":        item.get("summary", ""),
                        "reason":         item.get("reason", ""),
                        "recommendation": item.get("recommendation", "Review as needed")
                    })
                while len(results) < len(chunks):
                    results.append(_fallback_result(chunks[len(results)]))
                return results

        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "rate_limit" in err_str.lower():
                print(f"⚠️  Rate limit on {MODEL_POOL[current_model_idx]} — rotating model...")
                _rotate_model()
                time.sleep(2)  # brief pause before retry with new model
            else:
                print(f"⚠️  Batch error: {e}")
                break

    # All models exhausted — use keyword fallback
    print("⚠️  All models rate-limited — using keyword fallback for this batch")
    return [_fallback_result(c) for c in chunks]


def _fallback_result(chunk: str) -> Dict:
    cl = chunk.lower()
    if any(w in cl for w in ["irregular", "violation", "non-compliance", "late", "missing", "unauthorized"]):
        flag, rec = "Red",    "Immediate corrective action required"
    elif any(w in cl for w in ["compliant", "accordance", "proper", "timely", "approved"]):
        flag, rec = "Green",  "No action required"
    else:
        flag, rec = "Yellow", "Further investigation needed"
    return {
        "flag":           flag,
        "text":           chunk.strip(),
        "summary":        "Keyword-based classification (all models rate-limited)",
        "reason":         "Fallback — LLM unavailable",
        "recommendation": rec
    }

print("✅ Batch classifier ready")

✅ Batch classifier ready


In [46]:
def load_pdf_for_session(pdf_path: str, session_id: str, batch_size: int = 20):
    """Load PDF, classify all chunks in batches, return full grouped report"""

    print(f"🚀 Loading: {pdf_path}")
    start_time = time.time()

    loader   = PyPDFLoader(pdf_path)
    docs     = loader.load()
    # Smaller chunks = less tokens per batch call
    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
    chunks   = splitter.split_documents(docs)
    chunk_texts = [c.page_content for c in chunks]

    total_batches = (len(chunk_texts) + batch_size - 1) // batch_size
    print(f"📄 {len(docs)} pages → {len(chunk_texts)} chunks")
    print(f"🎯 API calls needed: {total_batches} (batch_size={batch_size})")

    db = FAISS.from_documents(chunks, embeddings)
    session_dbs[session_id]      = db
    session_histories[session_id] = ChatMessageHistory()
    flag_counts[session_id]       = {"Red": 0, "Yellow": 0, "Green": 0}

    all_results = []
    api_calls   = 0

    for i in range(0, len(chunk_texts), batch_size):
        batch     = chunk_texts[i:i + batch_size]
        batch_num = (i // batch_size) + 1
        print(f"⚡ Batch {batch_num}/{total_batches} ({len(batch)} chunks)...")

        results = classify_batch(batch)
        all_results.extend(results)
        api_calls += 1

        for r in results:
            flag_counts[session_id][r.get("flag", "Yellow")] += 1

        time.sleep(0.5)

    session_flags[session_id] = all_results

    total   = len(all_results)
    fc      = flag_counts[session_id]
    pct     = {k: round((v / total) * 100, 2) for k, v in fc.items()}
    elapsed = time.time() - start_time

    W = 72  # report width

    lines = []
    lines.append("=" * W)
    lines.append(" AUDIT ANALYSIS REPORT  (FAISS · all-MiniLM-L6-v2 · llama-3.1-8b-instant)")
    lines.append("=" * W)
    lines.append(f" ⏱  {elapsed:.1f}s   |   📄 {total} chunks   |   🔄 {api_calls} API calls   |   💰 {((total-api_calls)/total*100):.0f}% saved")
    lines.append("")
    lines.append(f" 🔴 Red    (Critical) : {fc['Red']:>4}  ({pct['Red']}%)")
    lines.append(f" 🟡 Yellow (Review)   : {fc['Yellow']:>4}  ({pct['Yellow']}%)")
    lines.append(f" 🟢 Green  (Compliant): {fc['Green']:>4}  ({pct['Green']}%)")
    lines.append("=" * W)

    for flag_type, emoji, label in [("Red","🔴","CRITICAL"),("Yellow","🟡","REVIEW"),("Green","🟢","COMPLIANT")]:
        items = [r for r in all_results if r.get("flag") == flag_type]
        if not items:
            lines.append(f"{emoji} {flag_type.upper()} FLAGS : none ✅")
            continue

        lines.append("")
        lines.append(f"{emoji}  {flag_type.upper()} FLAGS — {label}  ({len(items)} items)")
        lines.append("=" * W)

        for idx, r in enumerate(items, 1):
            lines.append(f"  [{flag_type.upper()}-{idx:03d}]")
            lines.append("  " + "─" * (W - 2))

            # Original text — wrap long lines cleanly
            raw = r['text'].replace("\n", " ").strip()
            lines.append("  📄 TEXT:")
            # word-wrap at ~66 chars
            words, line_buf = raw.split(), ""
            for word in words:
                if len(line_buf) + len(word) + 1 > 66:
                    lines.append("       " + line_buf)
                    line_buf = word
                else:
                    line_buf = (line_buf + " " + word).strip()
            if line_buf:
                lines.append("       " + line_buf)

            lines.append(f"  📋 SUMMARY        : {r['summary']}")
            lines.append(f"  🔍 REASON         : {r['reason']}")
            lines.append(f"  💡 RECOMMENDATION : {r['recommendation']}")
            lines.append("")

    lines.append("=" * W)
    lines.append(f" 🎯 Issues: {fc['Red']+fc['Yellow']}  |  Compliant: {fc['Green']}  |  Total: {total}")
    lines.append("=" * W)

    report = "\n".join(lines)
    return report

print("✅ PDF processor ready")

✅ PDF processor ready


In [47]:
def build_rag_chain(session_id: str):
    """Build conversational RAG chain using local FAISS only (no Pinecone)"""
    db = session_dbs.get(session_id)
    if not db:
        raise ValueError(f"No PDF loaded for session: {session_id}")

    retriever = db.as_retriever(search_kwargs={"k": 5})

    contextualize_q_prompt = ChatPromptTemplate.from_messages([
        ("system", "Given a chat history and the latest user question, formulate a standalone question. Do NOT answer."),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])
    history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

    qa_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a legal assistant specializing in Pakistani tax law.\n"
         "Use precise legal terminology and cite relevant law sections.\n\nContext:\n{context}"),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])

    rag_chain = create_retrieval_chain(
        history_aware_retriever,
        create_stuff_documents_chain(llm, qa_prompt)
    )

    return RunnableWithMessageHistory(
        rag_chain,
        lambda _: session_histories[session_id],
        input_messages_key="input",
        history_messages_key="chat_history",
        output_messages_key="answer",
    )


def query_audit(session_id: str, query: str) -> str:
    """Query the loaded PDF with conversational history"""
    if session_id not in session_dbs:
        return "⚠️ No PDF loaded. Run load_pdf_for_session() first."
    chain = build_rag_chain(session_id)
    result = chain.invoke(
        {"input": query},
        config={"configurable": {"session_id": session_id}}
    )
    return result["answer"]

print("✅ RAG chain ready")

✅ RAG chain ready


In [48]:
# ── RUN ──────────────────────────────────────────────────────────────────────
session_id = "session_001"
pdf_path   = r"D:\recovery\AGENT LEXA\Research\Tax Audit Report Format.pdf"  # ← update path

if os.path.exists(pdf_path):
    report = load_pdf_for_session(pdf_path, session_id, batch_size=8)
    print(report)

    # Optional: save report
    with open("audit_report.txt", "w", encoding="utf-8") as f:
        f.write(report)
    print("💾 Report saved to audit_report.txt")
else:
    print(f"❌ File not found: {pdf_path}")

🚀 Loading: D:\recovery\AGENT LEXA\Research\Tax Audit Report Format.pdf
📄 20 pages → 108 chunks
🎯 API calls needed: 14 (batch_size=8)
⚡ Batch 1/14 (8 chunks)...
⚠️  Batch error: Extra data: line 3 column 1 (char 275)
⚠️  All models rate-limited — using keyword fallback for this batch
⚡ Batch 2/14 (8 chunks)...
⚠️  Batch error: Extra data: line 3 column 1 (char 299)
⚠️  All models rate-limited — using keyword fallback for this batch
⚡ Batch 3/14 (8 chunks)...
⚡ Batch 4/14 (8 chunks)...
⚡ Batch 5/14 (8 chunks)...
⚡ Batch 6/14 (8 chunks)...
⚡ Batch 7/14 (8 chunks)...
⚡ Batch 8/14 (8 chunks)...
⚡ Batch 9/14 (8 chunks)...
⚡ Batch 10/14 (8 chunks)...
⚠️  Batch error: Extra data: line 3 column 1 (char 206)
⚠️  All models rate-limited — using keyword fallback for this batch
⚡ Batch 11/14 (8 chunks)...
⚡ Batch 12/14 (8 chunks)...
⚠️  Batch error: Extra data: line 3 column 1 (char 118)
⚠️  All models rate-limited — using keyword fallback for this batch
⚡ Batch 13/14 (8 chunks)...
⚡ Batch 14/14 (4

In [49]:
# load_pdf_for_session(session_id,pdf_path)

In [50]:
# ── QUERY (after loading PDF) ─────────────────────────────────────────────────
answer = query_audit(session_id, "What are the main violations found in this audit?")
print(answer)

Based on the provided audit report, the following main violations or discrepancies are reported:

1. **Discrepancies in the Balance Sheet and Profit/Loss Account**: The audit report mentions that the balance sheet and profit/loss account are not in agreement with the books of account maintained at the head office and branches. (Section 232 of the Companies Act, 2017)

2. **Non-Compliance with Central Excise Act, 1944**: The audit report indicates that an audit was conducted under the Central Excise Act, 1944, and there are disqualifications or disagreements on certain matters/items/values/quantities reported by the auditor. (Section 11 of the Central Excise Act, 1944)

3. **Non-Compliance with Cost Audit**: The audit report mentions that a cost audit was carried out, and there are disqualifications or disagreements on certain matters/items/values/quantities reported by the cost auditor. (Section 139(9) of the Companies Act, 2017)

4. **Non-Compliance with Section 44AB of the Income Tax